# LAION vs. COCO vocabulary: image-level concept intervention (Type 0)

In [ ]:
import sys
sys.path.append("..")

import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from utils import obj_enc_aggregation
from intervention_image_level import (
    build_laion_lexicon, load_intervention_testset, load_gt_categories, load_category_id_by_name,
    load_checkpoint, load_variants, load_boundary_type, apply_intervention, stepwise_intervention_curve,
    balanced_accuracy, positive_variant_accuracy, nb_accuracy,
)
from intervention_image_level_coco import build_trivial_lexicon

ENC = "../encodings/cocologic"
OBJ = "obj_7_crop_min_0.01_max_0.85_score_0.2_iou_0.5_model_mask-rcnn_0.25"
TRAIN_JSON = "../data/cocologic/cocologic_train.json"
TEST_JSON = "../data/cocologic/cocologic_test.json"
K = 7

VOCABS = {
    "COCO": {
        "lexicon": build_trivial_lexicon("../datasets/coco/annotations/instances_val2017.json"),
        "base": f"{ENC}/clip/splice_embeddings_0.25_coco_categories.h5",
        "obj": f"{ENC}/obj_encodings/clip/{OBJ}_coco_categories.h5",
        "ckpt": "../checkpoints/cocologic/sum_super_multilabel_seed42_coco_categories.pt",
        "targets": "../data/cocologic/percentile_targets_full_train_sum_coco_categories.pt",
    },
    "LAION": {
        "lexicon": build_laion_lexicon(),
        "base": f"{ENC}/clip/splice_embeddings_0.25.h5",
        "obj": f"{ENC}/obj_encodings/clip/{OBJ}.h5",
        "ckpt": "../checkpoints/cocologic/sum_super_multilabel_seed42.pt",
        "targets": "../data/cocologic/percentile_targets_full_train_sum_laion.pt",
    },
}

category_id_by_name = load_category_id_by_name(TRAIN_JSON)

RULE_NAMES = ["Signal and Ride", "Double Serving", "Herd Alone", "Dog or Car", "Three of a Kind",
              "Car Majority", "Empty Seat", "Single Mode Traffic", "Personal Transport", "Surf Trip"]

RULE_CATEGORIES = {
    "Signal and Ride": ["traffic light", "bicycle", "bus", "train"],
    "Double Serving": ["bottle", "cup", "pizza"],
    "Herd Alone": ["cow", "elephant", "sheep", "person"],
    "Dog or Car": ["dog", "car"],
    "Three of a Kind": ["bowl", "cup"],
    "Car Majority": ["car", "truck"],
    "Empty Seat": ["couch", "chair", "person"],
    "Single Mode Traffic": ["bicycle", "motorcycle", "car", "bus"],
    "Personal Transport": ["person", "bicycle", "car"],
    "Surf Trip": ["person", "surfboard"],
}
# relevant relevant categories 
RELEVANT = ["traffic light", "bicycle", "bus", "train", "bottle", "cup", "pizza", "cow", "elephant", "sheep",
            "person", "dog", "car", "bowl", "truck", "couch", "chair", "motorcycle", "surfboard"]
ALL_80 = list(VOCABS["COCO"]["lexicon"])

test_ids = None
for v in VOCABS.values():
    emb, obj, img_ids, labels = load_intervention_testset(v["base"], v["obj"], K)
    if test_ids is None:
        test_ids, test_labels = img_ids, labels.float()
    assert torch.equal(img_ids, test_ids)
    v["model"] = load_checkpoint(v["ckpt"])
    v["targets"] = torch.load(v["targets"])
    v["base_vec"] = obj_enc_aggregation("sum", obj, emb, only_objects=False)

gt_test = load_gt_categories(TEST_JSON, test_ids)  
variants = load_variants(TEST_JSON, test_ids)
boundary_type = load_boundary_type(TEST_JSON, test_ids)
for v in VOCABS.values():
    v["full_vec"] = apply_intervention(v["base_vec"], gt_test, v["targets"], category_id_by_name, suppress=True)

def rule_mean(model, x):
    with torch.no_grad():
        logits = model(x)
    return {
        "B-Acc": 100 * torch.nanmean(balanced_accuracy(logits, test_labels)).item(),
        "NB-Acc": 100 * torch.nanmean(nb_accuracy(logits, boundary_type)).item(),
        "PosVar-Acc": 100 * torch.nanmean(positive_variant_accuracy(logits, variants)).item(),
    }

In [ ]:
rows = []
for name, v in VOCABS.items():
    before = rule_mean(v["model"], v["base_vec"])
    after = rule_mean(v["model"], v["full_vec"])
    for m in before:
        rows.append({"vocab": name, "metric": m, "baseline": before[m], "intervention": after[m],
                     "difference": after[m] - before[m]})
print(pd.DataFrame(rows).round(2).to_string(index=False))

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
KEYS = {"B-Acc": "bacc", "NB-Acc": "nb", "PosVar-Acc": "pv"}

def permutations(categories):
    orders = []
    for seed in SEEDS:
        order = list(categories)
        random.Random(seed).shuffle(order)
        orders.append(order)
    return orders

ORDERS = {"19": permutations(RELEVANT), "80": permutations(ALL_80)}

stepwise = {}
for curve, orders in ORDERS.items():
    for name, v in VOCABS.items():
        stepwise[(curve, name)] = stepwise_intervention_curve(
            v["model"], v["base_vec"], gt_test, v["targets"], category_id_by_name,
            test_labels, variants, boundary_type, orders=orders, suppress=True,
        )

for curve in ORDERS:
    stds = [100 * stepwise[(curve, name)][1][key][1] for name in VOCABS for key in KEYS.values()]
    print(f"{curve} categories: std over 5 orders between {min(s.min() for s in stds):.2f} and {max(s.max() for s in stds):.2f} pp")

In [ ]:
STYLE = {"COCO": {"color": "#eb6834", "linestyle": "-"}, "LAION": {"color": "#2a78d6", "linestyle": "--"}}

for curve in ORDERS:
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharex=True)
    for ax, (m, key) in zip(axes, KEYS.items()):
        for name in VOCABS:
            ns, stats, _ = stepwise[(curve, name)]
            ax.plot(ns, 100 * stats[key][0], linewidth=2, label=name, **STYLE[name])
        ax.set_title(m)
        ax.set_xlabel("number of corrected categories")
        ax.grid(alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)
    axes[0].set_ylabel("mean over rules (%)")
    axes[0].legend(frameon=False)
    fig.tight_layout()
    fig.savefig(f"../figures/vocab_stepwise_{curve}.pdf")
    plt.show()

In [ ]:

delta = {}
for name in VOCABS:
    _, _, curves = stepwise[("19", name)]
    for m in ["B-Acc", "PosVar-Acc"]:
        per_cat = {}
        for order, values in zip(ORDERS["19"], curves[KEYS[m]]):
            for n, cat in enumerate(order, start=1):
                per_cat.setdefault(cat, []).append(100 * (values[n] - values[n - 1]))
        for cat, d in per_cat.items():
            delta[(name, m, cat)] = np.mean(d)

rows = []
for cat in RELEVANT:
    present = gt_test[:, category_id_by_name[cat]] > 0
    row = {"category": cat, "rules": sum(cat in cats for cats in RULE_CATEGORIES.values()),
           "images": int(present.sum())}
    for name, v in VOCABS.items():
        dims = v["lexicon"][cat]
        if not dims:
            row[f"{name} active (%)"] = "no group"
            continue
        # LAION: active if any group dimension > 0
        values = v["base_vec"][present][:, dims]
        active = (values > 0).any(dim=1)
        boost = sum(v["targets"][cat]["boost"].values())
        median = values[active].sum(dim=1).median().item() if active.any() else np.nan
        row[f"{name} active (%)"] = 100 * active.float().mean().item()
        if name == "COCO":
            row["COCO missing"] = int((~active).sum())
        row[f"{name} median"] = median
        row[f"{name} boost"] = boost
        row[f"{name} median / boost"] = median / boost if boost > 0 else np.nan
        row[f"{name} dB-Acc"] = delta[(name, "B-Acc", cat)]
        row[f"{name} dPosVar-Acc"] = delta[(name, "PosVar-Acc", cat)]
    rows.append(row)

category_table = pd.DataFrame(rows).set_index("category").sort_values("COCO dB-Acc", ascending=False)
pd.set_option("display.width", 300)
pd.set_option("display.max_columns", 30)
print(category_table.to_string(float_format=lambda x: f"{x:.3f}", na_rep=""))

In [ ]:
def dims_of(name, categories):
    return [d for cat in categories for d in VOCABS[name]["lexicon"][cat]]

for v in VOCABS.values():
    v["weight"] = v["model"].classifier.weight.detach()  

rows = []
for r, rule in enumerate(RULE_NAMES):
    row = {"rule": rule}
    for name, v in VOCABS.items():
        w = v["weight"][r].abs()
        total = w.sum().item()
        row[f"{name} (a) 19 relevant"] = 100 * w[dims_of(name, RELEVANT)].sum().item() / total
        row[f"{name} (b) rule's categories"] = 100 * w[dims_of(name, RULE_CATEGORIES[rule])].sum().item() / total
        row[f"{name} (c) 80 categories"] = 100 * w[dims_of(name, ALL_80)].sum().item() / total
    rows.append(row)
weight_share = pd.DataFrame(rows).set_index("rule")
weight_share.loc["mean"] = weight_share.mean()
print("share of sum |w| per rule (%)")
print(weight_share.round(2).to_string())

In [ ]:
rows = []
for cat in category_table.index:
    for rule in [r for r in RULE_NAMES if cat in RULE_CATEGORIES[r]]:
        r = RULE_NAMES.index(rule)
        row = {"category": cat, "rule": rule}
        for name, v in VOCABS.items():
            dims = v["lexicon"][cat]
            row[f"{name} weight"] = v["weight"][r, dims].sum().item() if dims else np.nan
        rows.append(row)
print("weight of the category concept in the rule (LAION: sum over the group)")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.3f}", na_rep="no group"))